# Проект 18 спринта: выявление рисков в доставке логистической компании CargaPronto

Выполнил: Артем Буров (DS12)  
Дата: 20/09/2026  
GitHub: https://github.com/TemaQDX/sprint_18_project_carga_pronto

## Описание проекта

Компания CargaPronto — крупнейший логистический оператор, связывающий производственные хабы Пуэрто-Рико с потребителями в США. В последнее время бизнес столкнулся с острым кризисом: 55 % заказов доставляются с задержкой, а объём штрафов за нарушение SLA за прошлый квартал превысил бюджет на развитие IT‑инфраструктуры. Крупнейшие ритейлеры США поставили ультиматум о необходимости стабилизации сроков доставки, иначе контракты будут расторгнуты.

Внутри компании нет единого понимания причин задержек: департамент транспорта связывает проблему с внешними факторами и неоднородностью клиентской базы — в частности, со сложными локациями и особенностями поведения отдельных потребителей. Представители отдела считают, что невозможно обеспечить одинаковые сроки доставки для всех клиентов из‑за существенных различий в их профилях.

Цель текущего проекта как независимого консультанта — проверить гипотезу о том, что задержки обусловлены не сбоями в логистике, а спецификой самих клиентов. Необходимо сегментировать клиентскую базу, выделить проблемные группы и доказать, что тип потребителя можно использовать для прогнозирования риска задержки ещё на этапе оформления заказа. Это позволит компании точечно корректировать приоритеты доставки и сократить объём штрафных выплат.

## Цель и задача проекта

**Цель проекта** — создать систему поддержки принятия решений для логистов CargaPronto, которая в момент оформления заказа в реальном времени оценивает риск задержки и выдаёт предупреждение, если вероятность опоздания превышает 70%. Получив такой сигнал, диспетчер или автоматизированная система управления складом может принять одно из нескольких корректирующих действий: повысить приоритет сборки заказа, за счёт компании перевести доставку в более быстрый режим (например, с Standard Class до First Class или Second Day), назначить на сложный маршрут более опытного курьера или дополнительный транспорт, заранее уведомить клиента о возможной задержке — что в логистике часто не считается нарушением лояльности, — либо перепроверить актуальность заказа с клиентом, склонным к возвратам. Такой подход позволяет не просто констатировать факт опоздания постфактум, а активно управлять ситуацией ещё до отгрузки, снижая объёмы штрафных выплат и удерживая лояльность клиентов.

**Задача проекта** — построить модель бинарной классификации, которая на основе входных данных о заказе и профиле клиента предсказывает вероятность задержки доставки. Положительный класс — заказ с задержкой, отрицательный — заказ, доставленный вовремя. Модель должна работать достаточно быстро, чтобы интеграция в процесс оформления заказа происходила в реальном времени.

В качестве инструментов предусмотрены два алгоритма. Логистическая регрессия выбрана как базовая модель: она ценится в бизнесе за высокую скорость и прозрачность, позволяет оценить «нижнюю планку» качества до применения более сложных методов. В качестве основной модели используется градиентный бустинг CatBoost — он способен находить сложные нелинейные закономерности в данных и эффективно работать с категориальными признаками, что особенно важно при сегментировании клиентской базы.

Основная метрика качества — ROC-AUC. Выбор обусловлен тем, что метрика оценивает способность модели ранжировать заказы по степени риска, а не просто порогово классифицировать их. Это позволяет диспетчерам CargaPronto точечно выделять те доставки, которым действительно нужно больше внимания, и гибко управлять порогом срабатывания предупреждений. Удовлетворительным результатом для проекта считается значение ROC-AUC выше 0,75.

Внедрение такой системы переводит CargaPronto от реактивной модели работы — выплаты штрафов по факту опоздания — к проактивной: компания получает возможность предотвращать задержки до того, как заказ покинет склад, и управлять ресурсами исходя из прогнозируемого риска, а не ограничиваться разбором причин уже случившихся срывов.


## Описание данных

### Датасет профилей клиентов (`ds_s18_customers.csv`)

Агрегированная информация о каждом уникальном клиенте из ERP-системы. Признаки характеризуют пользователей с географической и поведенческой сторон. Поведенческие признаки собраны по методологии RFM — стандарту сегментации в ритейле и логистике, позволяющему отделить лояльных клиентов от случайных покупателей. Помимо базовых RFM-показателей, датасет содержит данные о средней скидке и доле возвратов. На основе географических координат и поведенческих метрик планируется создание новых признаков с помощью обучения без учителя: геокластеров (группировка клиентов по реальным логистическим зонам) и поведенческих сегментов (выделение групп на основе RFM, скидки и возвратов).

| Признак | Описание |
|---|---|
| `customer_id` | Уникальный номер клиента в ERP. Ключ для связи с таблицей заказов. |
| `customer_lat` | Широта местонахождения клиента. |
| `customer_lon` | Долгота местонахождения клиента. |
| `recency` | Количество дней с момента последнего заказа до точки отсчёта (Recency). |
| `total_orders` | Общее количество заказов клиента (Frequency). |
| `total_sales` | Суммарная выручка от клиента за всё время (Monetary). |
| `avg_discount` | Средний размер скидки, получаемой клиентом. Маркер чувствительности к акциям. |
| `return_rate` | Доля проблемных заказов (возвраты и отмены). Ключевой показатель добросовестности клиента. |

### Датасет заказов (`ds_s18_orders.csv`)

Таблица событий, где зафиксирован каждый факт продажи и итог доставки. Именно здесь находится целевая переменная проекта — `late_delivery_risk`. Каждая строка описывает отдельный заказ: дату совершения, категорию товара, стоимость и выбранный режим доставки. Заказы связаны с профилями клиентов через `customer_id`, что позволяет объединять поведенческие и географические характеристики клиента с параметрами конкретного заказа для предсказания риска задержки.

| Признак | Описание |
|---|---|
| `order_id` | Уникальный номер заказа. |
| `customer_id` | Идентификатор клиента. Ключ для связи с профилем из таблицы клиентов. |
| `order_date` | Дата и время совершения покупки. |
| `category_name` | Категория заказанного товара. |
| `item_price` | Стоимость товара в заказе. |
| `shipping_mode` | Выбранный режим доставки: Standard Class, First Class, Second Class, Same Day. |
| `late_delivery_risk` | Целевая переменная: 1 — заказ доставлен с задержкой, 0 — доставлен вовремя. |


##  Импорт библиотек, выполнение базовых настроек

Загрузите все необходимые для выполнения проекта библиотеки и другие компоненты.



## Загрузка данных, первичное знакомство

* Загрузите датасеты `ds_s18_customers.csv` и `ds_s18_orders.csv`.
  * Путь к первому файлу — `'/datasets/ds_s18_customers.csv'`.
  * Путь ко второму файлу — `'/datasets/ds_s18_orders.csv'`.
* Проведите технический аудит данных.
* Выявите и устраните полные дубликаты строк.
* Выполните проверку целостности данных (ID-Check).
* Подготовьте признаки, связанные с датой и временем.
* Создайте дополнительные колонки `order_month`, `order_weekday`, `order_hour`.




## EDA

На этом этапе ваша цель — исследовать структуру данных и подтвердить гипотезы транспортного департамента о «неоднородности» базы. Кроме того, вам предстоит изучить и удалить географические аномалии.

1. Проверьте баланс классов.
2. Выполните географический аудит.
3. Удалите аномалии.
4. Проанализируйте зависимости между признаками.



##  Разделение на выборки

Разделите датафрейм с заказами на выборки. В этом проекте вам предстоит  использовать «классическое» разделение на три выборки: обучающую, валидационную и тестовую. Использовать для этого нужно не `train_test_split`, а `GroupShuffleSplit`. Он делит данные так, чтобы группы, то есть клиенты, не пересекались.

Ниже приведён пример того, как отделить часть данных для обучения, используя `customer_id` как ключ для группировки.

```python
from sklearn.model_selection import GroupShuffleSplit

# 1. Выбираем колонку, по которой будем группировать (наши "группы")
groups = df_orders['customer_id']

# 2. Инициализируем сплиттер (например, отделим 20% клиентов для теста)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)

# 3. Получаем индексы для разделения
# Метод split возвращает генератор, поэтому используем next()
train_idx, test_idx = next(gss.split(df_orders, groups=groups))

# 4. Формируем выборки
df_train = df_orders.iloc[train_idx]
df_test = df_orders.iloc[test_idx]

```


1. Разделите данные из датафрейма с заказами  на три выборки  в соотношении 60:20:20.


2. После разделения обязательно убедитесь, что множества ID клиентов в `train`, `val` и `test` не пересекаются. Если один и тот же клиент попал в разные выборки, то разделение выполнено неверно.

## Обучение базовой модели

Прежде чем проверять гипотезу о сегментации и создавать при помощи кластеризации новые признаки, необходимо зафиксировать точку отсчёта. Вам нужно понять, какое качество прогноза задержек обеспечивают стандартные модели, используя только базовую информацию о самом заказе.

1. Выберите нужные признаки — используйте колонки таблицы `orders` (`shipping_mode`, `category_name`, `item_price`, `order_hour`,  `order_weekday` и `order_month`).

2. Обучите логистическую регрессию и `CatBoostClassifier` и оцените их качество.

3. Оцените качество обеих моделей на валидационной выборке.

Напоминаем, что на этапе построения базовой модели глубокий подбор гиперпараметров необязателен.

**Дополнительное задание.** Если вы чувствуете в себе силы, вы можете попробовать базовую оптимизацию, но помните: главная прибавка к качеству ожидается от новых признаков, которые вы создадите в следующих разделах.






##  Кластеризация по признакам, связанным с местоположением



На этом этапе нужно превратить исходные координаты `customer_lat` и `customer_lon` в полезный для модели признак — логистическую зону.

**Ваши действия:**

1. Проведите корректную подготовку признаков.
2. Продумайте защиту от утечки данных.
3. Используйте метод локтя, чтобы найти математически обоснованную границу количества групп.
4. Постройте график с полученными кластерами и отметьте центроиды.

##  Кластеризация по признакам профилей клиентов





1. Подготовьте векторы признаков — используйте пять ключевых показателей из датасета `df_customers.csv`: `recency`, `total_orders`, `total_sales`, `return_rate` и `avg_discount`.
2. Выполните предобработку данных и обеспечьте защиту от утечек.
3. Используйте метод локтя, чтобы найти математически обоснованную границу количества групп.
4. Проанализируйте полученные кластеры: дайте статистическую характеристику и опишите самые яркие группы.
5. Визуализируйте положение кластеров с помощью t-SNE.



##  Обучение модели на новых признаках

1. Добавьте результаты кластеризаций в данные для обучения моделей.
2. Обучите финальные версии логистической регрессии и `CatBoostClassifier` на расширенном наборе данных.
3. Рассчитайте итоговые значения метрики ROC−AUC на валидационной выборке.
   

## Дополнительное задание — подбор лучшего количества кластеров

Рассмотрите количество кластеров как гиперпараметр всей системы и найти ту «степень детализации», которая даст максимальный прирост ROC-AUC.

Вы можете взять значения из списка или выбрать свои:

```python
geo_k_range = [4, 8, 12, 20]
rfm_k_range = [6, 12, 20, 30]
```



## Тестирование лучшей модели

На этом этапе вы должны убедиться, что выбранная модель сохраняет высокое качество на данных, которые она никогда не видела, и понять, какие факторы стали решающими для прогноза.

1. Выполните предсказание на тестовой выборке для лучшей модели. Рассчитайте итоговый ROC-AUC и сравните его с целевым показателем 0.75.
2. Постройте матрицу ошибок. Определите, какой тип ошибок совершает модель чаще: пропускает ли она реальные задержки или слишком часто выдаёт ложную тревогу?
3. Визуализируйте важность признаков вашей лучшей модели.
4. Проанализируйте позиции созданных вами признаков `geo_cluster` и `beh_cluster` в общем рейтинге. Стали ли они ключевыми факторами для предсказаний модели для модели или базовые параметры заказа (цена, время) остались приоритетными?

##  Выводы и рекомендации

Что нужно зафиксировать:

* **Результаты моделирования.** Укажите итоговое значение ROC-AUC на тестовой выборке. Удалось ли вам достичь целевого показателя? Насколько эффективно итоговая модель справляется с выявлением задержек по сравнению с базовыми?
* **Эффективность сегментации.** Сформулируйте вывод о полезности кластеризации. Подтвердилась ли гипотеза о том, что профиль клиента и его географическое положение влияют на риск задержки? Какие именно кластеры — географические или поведенческие — оказались более информативными для модели? Добавьте в раздел визуализации, которые вы получили, работая над проектом.
* **Технический инсайт.** Если вы экспериментировали с разным количеством  кластеров как с гиперпараметром, то укажите, какое количество кластеров K оказалось оптимальным. Кратко поясните, почему слишком большое K может вредить качеству прогноза.
* **Бизнес-рекомендации.** Предложите, как CargaPronto может использовать вашу модель в реальных операциях.